# Train GGPKD Top-K-only, no configured negatives (Colab)

Notebook ablation: use exactly `TOPK_QUOTA` candidate slots, all assigned to graph Top-K support (`hard_neg_k=0`, `random_neg_k=0`).

In [1]:
#@title 1. Cấu hình experiment
REPO_URL = "https://github.com/Savoxism/embedding-kd.git"
BRANCH = "nqd_ablation"

PAIR_KEY = "qwen3_4b_to_bert_base" #@param
# ["qwen3_0_6b_to_minilmv2_h384", "bge_m3_to_minilmv2_h768", "qwen3_4b_to_bert_base"]

# Ablation Top-K-only: candidate width = TOPK_QUOTA + 0 hard + 0 random.
# TOPK_QUOTA được truyền explicit, không dùng quota tự suy ra từ graph.
TOPK_QUOTA = 81 #@param {type:"integer"}
HARD_NEG_K = 0
RANDOM_NEG_K = 0
SUPPORT_POLICY = "topk"
DIFFUSION_SCALES = "1"  # paper protocol R={1}

ROW_WEIGHT = 1.0 #@param {type:"number"}
ROW_START_EPOCH = 1 #@param {type:"integer"}

RUN_NAME = "topk_no_neg" #@param {type:"string"}

BATCH_SIZE = 64 #@param {type:"integer"}
EPOCHS = 5 #@param {type:"integer"}
# 3e-5 là arm thắng từ config; run 77.89 trước đó chạy 2e-5 và có chữ ký
# undertrain rõ (loss còn giảm ở epoch 4→5, student_top1 còn tăng).
LEARNING_RATE = 3e-5 #@param {type:"number"}
MAX_LENGTH = 256 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
NUM_WORKERS = 4 #@param {type:"integer"}
TRAIN_DATA = "data/train_set/merged_3_data_5k_each.csv" #@param {type:"string"}

FINAL_WEIGHTS_ONLY = True #@param {type:"boolean"}
REQUIRE_GPU = True #@param {type:"boolean"}

USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/embedding-kd-runs" #@param {type:"string"}

# Thêm CLI flags không liên quan allocation nếu cần. Các flag Top-K/negative/R
# được khóa trong cell train để tránh EXTRA_ARGS vô tình override ablation.
EXTRA_ARGS = "--direct_temp 0" #@param {type:"string"}

assert ROW_WEIGHT >= 0, "ROW_WEIGHT phải không âm"
assert ROW_START_EPOCH >= 1, "ROW_START_EPOCH phải bắt đầu từ 1"
assert TOPK_QUOTA > 0, "TOPK_QUOTA phải dương"
assert HARD_NEG_K == 0 and RANDOM_NEG_K == 0, "Notebook này khóa negatives bằng 0"
assert DIFFUSION_SCALES == "1", "No-negative arm phải giữ canonical R={1}"
assert BATCH_SIZE > 0 and EPOCHS > 0 and MAX_LENGTH > 0

# Mỗi quota/seed ghi vào thư mục riêng để không đè lên run khác.
RUN_TAG = (
    f"{RUN_NAME}_w{ROW_WEIGHT:g}_e{ROW_START_EPOCH}"
    f"_r1_topk{TOPK_QUOTA}_h0_n0_lr{LEARNING_RATE:g}_seed{SEED}"
)


In [2]:
# 2. Clone lần đầu; các lần Run all sau luôn fetch/reset/pull branch mới nhất
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/embedding-kd")

def git(*args):
    command = ["git", "-C", str(REPO_DIR), *args]
    print("+", " ".join(command))
    subprocess.run(command, check=True)

if (REPO_DIR / ".git").is_dir():
    git("remote", "set-url", "origin", REPO_URL)
    # Bỏ thay đổi tracked trong clone Colab để luôn checkout được remote head.
    git("reset", "--hard")
    git("fetch", "--prune", "origin")
    git("checkout", "-B", BRANCH, f"origin/{BRANCH}")
    git("reset", "--hard", f"origin/{BRANCH}")
    git("pull", "--ff-only", "origin", BRANCH)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} tồn tại nhưng không phải Git repository")
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print(f"Ready: {BRANCH}@{commit[:12]}")


+ git -C /content/embedding-kd remote set-url origin https://github.com/Savoxism/embedding-kd.git
+ git -C /content/embedding-kd reset --hard
+ git -C /content/embedding-kd fetch --prune origin
+ git -C /content/embedding-kd checkout -B nqd_ablation origin/nqd_ablation
+ git -C /content/embedding-kd reset --hard origin/nqd_ablation
+ git -C /content/embedding-kd pull --ff-only origin nqd_ablation
Ready: nqd_ablation@80d083258245


In [3]:
# 3. Cài/đồng bộ dependencies theo code vừa pull
import os
import sys

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
print("Dependencies are ready.")


Dependencies are ready.


In [4]:
# 4. Resolve model pair, GPU và nơi lưu artifacts
import torch

PAIR_CONFIGS = {
    "qwen3_0_6b_to_minilmv2_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "pooling": "last_token",
    },
    "bge_m3_to_minilmv2_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "pooling": "cls",
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "pooling": "last_token",
    },
}
pair = PAIR_CONFIGS[PAIR_KEY]

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Không tìm thấy CUDA GPU. Trong Colab chọn Runtime → Change runtime type → GPU.")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 2**30
    print(f"GPU: {props.name} ({vram_gb:.1f} GiB)")
    if PAIR_KEY == "qwen3_4b_to_bert_base" and vram_gb < 35:
        print("WARNING: Qwen3-4B ở cấu hình hiện tại có thể OOM trên GPU dưới khoảng 35 GiB.")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    artifact_root = Path(DRIVE_ROOT)
else:
    artifact_root = REPO_DIR

# Teacher cache dùng chung; graph cache mang r1 để không tái dùng nhầm artifact
# multi-hop từ notebook gốc.
cache_dir = artifact_root / "cache" / "ggpkd" / PAIR_KEY
log_dir = artifact_root / "logs" / "ggpkd" / PAIR_KEY
save_dir = artifact_root / "models" / "ggpkd" / PAIR_KEY / RUN_TAG
weights_dir = artifact_root / "models" / "ggpkd_weights" / PAIR_KEY / RUN_TAG
for path in (cache_dir, log_dir, save_dir, weights_dir):
    path.mkdir(parents=True, exist_ok=True)

print(f"Teacher: {pair['teacher']}")
print(f"Student: {pair['student']}")
print(f"Pooling: {pair['pooling']}")
print(f"Objective: L_rel + {ROW_WEIGHT} * L_row")
print(f"Row starts at epoch: {ROW_START_EPOCH}")
print(f"Diffusion scales: {DIFFUSION_SCALES}")
print(f"Candidate allocation: topk={TOPK_QUOTA}, hard=0, random=0")
print(f"Candidate width: {TOPK_QUOTA}")
print(f"Outputs: {save_dir}")


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
Teacher: Qwen/Qwen3-Embedding-4B
Student: google-bert/bert-base-uncased
Pooling: last_token
Objective: L_rel + 1.0 * L_row
Row starts at epoch: 1
Diffusion scales: 1
Candidate allocation: topk=81, hard=0, random=0
Candidate width: 81
Outputs: /content/embedding-kd/models/ggpkd/qwen3_4b_to_bert_base/topk_no_neg_w1_e1_r1_topk81_h0_n0_lr3e-05_seed42


In [ ]:
# 5. Train GGPKD
import shlex

extra_args = shlex.split(EXTRA_ARGS)
locked_flags = {"--support_policy", "--diffusion_quota", "--diffusion_scales",
                "--hard_neg_k", "--random_neg_k"}
overrides = sorted(locked_flags.intersection(extra_args))
if overrides:
    raise ValueError(f"EXTRA_ARGS không được override các flag đã khóa: {overrides}")

command = [
    sys.executable, "main.py",
    "--method", "ggpkd",
    "--train_data", TRAIN_DATA,
    "--student_model", pair["student"],
    "--teacher_model", pair["teacher"],
    "--pooling_method", pair["pooling"],
    "--support_policy", SUPPORT_POLICY,
    "--diffusion_quota", str(TOPK_QUOTA),
    "--diffusion_scales", DIFFUSION_SCALES,
    "--hard_neg_k", str(HARD_NEG_K),
    "--random_neg_k", str(RANDOM_NEG_K),
    "--row_weight", str(ROW_WEIGHT),
    "--row_start_epoch", str(ROW_START_EPOCH),
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--eval_every", "0",
    "--lr", str(LEARNING_RATE),
    "--max_length", str(MAX_LENGTH),
    "--seed", str(SEED),
    "--num_workers", str(NUM_WORKERS),
    "--cache_path", str(cache_dir / "teacher_train.pt"),
    "--ggpkd_cache_path", str(cache_dir / "graph_r1.pt"),
    "--ggpkd_log_dir", str(log_dir),
    "--save_dir", str(save_dir),
    "--weights_dir", str(weights_dir),
]
if FINAL_WEIGHTS_ONLY:
    command.append("--final_weights_only")
if extra_args:
    command.extend(extra_args)

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("+", shlex.join(command))
process = subprocess.Popen(
    command,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")


+ /usr/bin/python3 main.py --method ggpkd --train_data data/train_set/merged_3_data_5k_each.csv --student_model google-bert/bert-base-uncased --teacher_model Qwen/Qwen3-Embedding-4B --pooling_method last_token --support_policy topk --diffusion_quota 81 --diffusion_scales 1 --hard_neg_k 0 --random_neg_k 0 --row_weight 1.0 --row_start_epoch 1 --batch_size 64 --epochs 5 --eval_every 0 --lr 3e-05 --max_length 256 --seed 42 --num_workers 4 --cache_path /content/embedding-kd/cache/ggpkd/qwen3_4b_to_bert_base/teacher_train.pt --ggpkd_cache_path /content/embedding-kd/cache/ggpkd/qwen3_4b_to_bert_base/graph_r1.pt --ggpkd_log_dir /content/embedding-kd/logs/ggpkd/qwen3_4b_to_bert_base --save_dir /content/embedding-kd/models/ggpkd/qwen3_4b_to_bert_base/topk_no_neg_w1_e1_r1_topk81_h0_n0_lr3e-05_seed42 --weights_dir /content/embedding-kd/models/ggpkd_weights/qwen3_4b_to_bert_base/topk_no_neg_w1_e1_r1_topk81_h0_n0_lr3e-05_seed42 --final_weights_only --direct_temp 0

Configuration for GGPKD method:
  

In [ ]:
# 6. Xem artifacts và các metrics cuối
import json

print("Checkpoints:", save_dir)
print("Weights:", weights_dir)
metrics_path = save_dir / "metrics.jsonl"
if metrics_path.exists():
    records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    print(json.dumps(records[-1], indent=2, ensure_ascii=False) if records else "metrics.jsonl is empty")
else:
    print("Không tìm thấy metrics.jsonl")

In [ ]:
# 7. Phân tích run: run.json / metrics.jsonl / epochs.jsonl / step_metrics.jsonl
# Các file đều append-mode, nên mọi thứ được lọc theo run_id của run mới nhất
# (run.json bị ghi đè mỗi lần train nên luôn trỏ vào run vừa chạy).
import json
from pathlib import Path

import pandas as pd

RUN_DIR = Path(save_dir)  # save_dir đến từ cell 4; sửa tay nếu chạy cell này rời


def read_jsonl(path, run_id=None):
    path = Path(path)
    if not path.exists():
        return []
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    if run_id is not None:
        rows = [r for r in rows if r.get("run_id") == run_id]
    return rows


manifest = json.loads((RUN_DIR / "run.json").read_text())
run_id = manifest["run_id"]
cfg = manifest.get("config", {})
git = manifest.get("git", {})
graph = manifest.get("artifact", {}).get("graph_stats", {})

print(f"run_id      : {run_id}")
print(f"git         : {git.get('branch')}@{str(git.get('sha'))[:12]}"
      f"{' (DIRTY — sha không tái lập được run này)' if git.get('dirty') else ''}")
print("config      : " + ", ".join(
    f"{k}={cfg.get(k)}" for k in
    ("support_policy", "diffusion_scales", "diffusion_quota",
     "hard_neg_k", "random_neg_k",
     "row_weight", "row_start_epoch", "row_mode", "learning_rate",
     "epochs", "batch_size", "seed") if k in cfg))
expected_allocation = {
    "support_policy": SUPPORT_POLICY,
    "diffusion_quota": TOPK_QUOTA,
    "hard_neg_k": 0,
    "random_neg_k": 0,
}
mismatch = {k: (cfg.get(k), expected) for k, expected in expected_allocation.items()
            if cfg.get(k) != expected}
if mismatch:
    raise RuntimeError(f"Run không dùng đúng Top-K-only allocation: {mismatch}")
if tuple(cfg.get("diffusion_scales", ())) != (1,):
    raise RuntimeError(f"Run không dùng canonical R={{1}}: {cfg.get('diffusion_scales')}")
if graph:
    print("graph_stats : " + ", ".join(f"{k}={v}" for k, v in sorted(graph.items())))

# ---- Bảng per-epoch: train means + geometry probe -------------------------
epochs = read_jsonl(RUN_DIR / "epochs.jsonl", run_id)
if epochs:
    rows = []
    for r in epochs:
        flat = {"epoch": r.get("epoch")}
        flat.update({k: v for k, v in (r.get("train") or {}).items()
                     if isinstance(v, (int, float))})
        flat.update({f"geo_{k}": v for k, v in (r.get("geometry") or {}).items()})
        if isinstance(r.get("test"), dict):
            flat["test_avg"] = r["test"].get("avg")
        rows.append(flat)
    df_epochs = pd.DataFrame(rows).set_index("epoch")

    train_cols = [c for c in (
        "loss", "loss_rel", "loss_row_weighted", "row_count", "row_exposed_mass",
        "row_valid_ratio", "target_entropy", "student_entropy",
        "student_entropy_ratio", "student_top1", "target_top1", "js_floor",
        "grad_norm", "mean_step_seconds") if c in df_epochs.columns]
    geo_cols = [c for c in (
        "geo_anisotropy", "geo_cos_std", "geo_effective_rank", "geo_uniformity",
        "geo_alignment", "geo_positive_cos_mean", "geo_separation",
        "geo_teacher_student_spearman", "geo_teacher_anisotropy") if c in df_epochs.columns]

    print("\n=== Train means theo epoch ===")
    display(df_epochs[train_cols].round(4))
    if geo_cols:
        print("=== Geometry probe theo epoch ===")
        display(df_epochs[geo_cols].round(4))
else:
    df_epochs = pd.DataFrame()
    print("\nKhông có dòng nào trong epochs.jsonl cho run này.")

# ---- Kết quả test cuối (eval_every=0 nên chỉ có một bản ở cuối run) --------
final = next((r for r in reversed(read_jsonl(RUN_DIR / "metrics.jsonl", run_id))
              if r.get("test")), None)
if final:
    t = final["test"]
    print(f"=== Test cuối ===  avg={t['avg']}  avg_in={t['avg_in']}  avg_out={t['avg_out']}")
    per_task = {}
    for domain in ("classification", "pair", "sts"):
        for path, res in (t.get(domain) or {}).items():
            name = Path(path).stem.replace("_test", "")
            if domain == "sts":
                per_task[name] = {"domain": "sts", "score": round(100 * res, 2)}
            else:
                key = "f1" if domain == "classification" else "average_precision"
                per_task[name] = {"domain": domain, "score": round(100 * res[key], 2),
                                  "f1": round(100 * res["f1"], 2)}
    display(pd.DataFrame(per_task).T)
else:
    print("Chưa có record test nào trong metrics.jsonl cho run này.")


In [ ]:
# 8. Đồ thị step-level từ step_metrics.jsonl
import matplotlib.pyplot as plt

steps = read_jsonl(RUN_DIR / "step_metrics.jsonl", run_id)
if not steps:
    raise SystemExit("Không có step_metrics.jsonl cho run này.")

df = pd.DataFrame(steps).sort_values("global_step").set_index("global_step")
smooth = lambda s, w=50: s.rolling(w, min_periods=1).mean()

panels = [
    ("Loss (rolling mean 50 step)",
     [("loss", "total"), ("loss_rel", "L_rel"), ("loss_row_weighted", "w·L_row")]),
    ("Row diagnostics",
     [("row_exposed_mass", "row_exposed_mass"), ("row_valid_ratio", "row_valid_ratio")]),
    ("Entropy (student vs target)",
     [("student_entropy", "student"), ("target_entropy", "target"),
      ("student_entropy_ratio", "ratio")]),
    ("LR & grad norm",
     [("lr_next", "lr"), ("grad_norm", "grad_norm")]),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
epoch_starts = df.groupby("epoch").apply(lambda g: g.index.min())
for ax, (title, series) in zip(axes.flat, panels):
    drawn = False
    for col, label in series:
        if col in df.columns and df[col].notna().any():
            ax.plot(smooth(df[col]), label=label, linewidth=1)
            drawn = True
    for e, s in epoch_starts.items():
        ax.axvline(s, color="gray", alpha=0.25, linewidth=0.8)
    ax.set_title(title, fontsize=10)
    if drawn:
        ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
fig.suptitle(f"{run_id} — vạch xám dọc = ranh giới epoch", fontsize=11)
fig.tight_layout()
plt.show()

# Nhìn nhanh: loss step đầu/cuối mỗi epoch, để thấy còn giảm hay đã phẳng
ep_summary = df.groupby("epoch")["loss"].agg(["first", "last", "mean"]).round(4)
ep_summary["drop_in_epoch"] = (ep_summary["first"] - ep_summary["last"]).round(4)
display(ep_summary)


In [ ]:
# 9. Kiểm tra support availability và exposure của Top-K-only từ graph.pt
# Mixture row của mỗi anchor nằm sẵn trong artifact; cumsum theo mass giảm dần
# cho thẳng đường cong "quota k thì exposure bao nhiêu" và trần khi quota→∞.
import numpy as np
import torch

art = torch.load(str(cache_dir / "graph_r1.pt"), map_location="cpu")
pool_probs = art["pool_probs"].numpy()                 # (n_scales, n_items, width)
scales = art["metadata"]["diffusion_scales"]
w = np.array([1.0 / r for r in scales]); w /= w.sum()  # omega_r = 1/r, như sampler
mix = (pool_probs * w.reshape(-1, 1, 1)).sum(0)
cum = np.cumsum(-np.sort(-mix, axis=1), axis=1)
positive_support = (mix > 0).sum(axis=1)
short_rows = positive_support < TOPK_QUOTA
print(f"Top-K-only yêu cầu K={TOPK_QUOTA}; anchor thiếu K support dương: "
      f"{short_rows.sum()}/{short_rows.size} ({short_rows.mean():.2%}).")
if short_rows.any():
    print("WARNING: sampler hiện tại sẽ backfill các slot thiếu bằng uniform nodes; "
          "hard_neg_k=random_neg_k=0 không đảm bảo tuyệt đối mọi candidate đều có diffusion mass.")


ceiling = cum[:, -1]
print(f"Trần exposure (quota→∞, do graph_k + truncation quyết định): "
      f"mean={ceiling.mean():.3f}  p10={np.quantile(ceiling, 0.1):.3f}")
print(f"{'quota':>6} {'mean':>7} {'p10':>7} {'p50':>7} {'p90':>7}")
probe_quotas = sorted(set((TOPK_QUOTA, 8, 11, 15, 23, 30, 38, 45, 64, 90, 128)))
for q in probe_quotas:
    m = cum[:, min(q, cum.shape[1]) - 1]
    print(f"{q:6d} {m.mean():7.3f} {np.quantile(m, 0.1):7.3f} "
          f"{np.quantile(m, 0.5):7.3f} {np.quantile(m, 0.9):7.3f}")

# Quota nhỏ nhất đạt coverage tau cho từng anchor — đọc ngược bảng trên.
for tau in (0.5, 0.6, 0.7, 0.8):
    reached = ceiling >= tau
    need = (cum >= tau).argmax(axis=1) + 1
    need = need[reached]
    print(f"tau={tau}: quota cần p50={np.median(need):.0f}  "
          f"p90={np.quantile(need, 0.9):.0f}  "
          f"({(~reached).mean():.1%} anchor không bao giờ đạt trần này)")
